# CS781 Capstone - Phase 6: Abstractive Legal Summarization
Name: Aditya Kumar | Roll No: 240059 | Email: adityaj24@iitk.ac.in

In [1]:
from collections import Counter
import os, re, gc, glob, math, json, time, random, shutil, zipfile, hashlib, subprocess, sys, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(DEVICE, NUM_GPUS)

cuda 2


In [2]:
import os
PLAN = [
    ("lightweight", [dict(method="pfeiffer"), dict(method="kd")]),
]
EXP_TAG = "p6"
SMOKE = False

hf_token = os.environ.get("HF_TOKEN")
HF_REPO, P6_DIR = "Exploration-Lab/CS-781-Capstone", "Phase_6_Summ"
LOCAL_DATA_DIR = None
INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
OUT_ROOT = os.environ.get("CS781_OUT_ROOT", "/kaggle/working")

INPUT_MODE = "select"
SRC_TOK_LM, TGT_TOK_LM = 256, 254
SRC_TOK_ED, TGT_TOK_ED = 480, 150
DEV_FRAC, DEV_MIN = 0.05, 40

EPOCHS = {"houlsby": 6, "pfeiffer": 6, "lora": 6, "qlora": 6, "lorma": 6, "kd": 6, "full": 4, "partial": 5, "bitfit": 8}
LR     = {"houlsby": 5e-4, "pfeiffer": 5e-4, "lora": 1e-3, "qlora": 1e-3, "lorma": 5e-4, "kd": 2e-4, "full": 1e-4, "partial": 1e-4, "bitfit": 1e-3}
WEIGHT_DECAY, WARMUP_FRAC, PATIENCE, LABEL_SMOOTH = 0.01, 0.06, 2, 0.05
BATCH_SIZE, GRAD_ACCUM = 8, 2
TRAIN_DOC_LIMIT = None
RUN_MAX_MIN = 90
PEFT = dict(r=16, alpha=32, targets=("q_proj", "k_proj", "v_proj", "out_proj"), lorma_targets=("q_proj", "v_proj"), bottleneck=64, partial_last=4)
KD_T, KD_ALPHA, STUDENT_KEEP_EVERY = 2.0, 0.5, 2

LOG_EVERY = 25
GEN_BATCH, MIN_NEW, NO_REPEAT_NGRAM, REP_PENALTY = 16, 30, 3, 1.0
DEV_BERTSCORE = False

if SMOKE: EXP_TAG = "p6smoke"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("plan:", [(a, [r["method"] for r in rs]) for a, rs in PLAN])

plan: [('lightweight', ['pfeiffer', 'kd'])]


In [3]:
ROPE_BASE = 10000.0

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class DecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_seq_len:])
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

class MultiHeadAttention(nn.Module):
    """Generic multi-head attention. causal=True masks future query->key positions (self-attn
    in the decoder). query and key/value can come from different sequences (cross-attention)."""
    def __init__(self, d_model, n_heads, dropout=0.0, causal=False, max_seq_len=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.causal = causal
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if causal:
            mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
            self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x_q, x_kv):
        B, T, C = x_q.shape
        S = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if self.causal:
            scores = scores.masked_fill(~self.causal_mask[:T, :S], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

def feed_forward(d_model, d_ff, dropout):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_tgt_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=True, max_seq_len=max_tgt_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x, enc_out):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_out)
        x = x + self.ff(self.ln3(x))
        return x

class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff,
                 max_src_len, max_tgt_len, dropout=0.1):
        super().__init__()
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("enc_pos", sinusoidal_positional_encoding(max_src_len, d_model), persistent=False)
        self.register_buffer("dec_pos", sinusoidal_positional_encoding(max_tgt_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.encoder_blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.decoder_blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, max_tgt_len, dropout) for _ in range(n_dec_layers)])
        self.enc_ln_f = nn.LayerNorm(d_model)
        self.dec_ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def encode(self, src_ids):
        T = src_ids.shape[1]
        x = self.drop(self.token_emb(src_ids) + self.enc_pos[:T].unsqueeze(0))
        for block in self.encoder_blocks:
            x = block(x)
        return self.enc_ln_f(x)

    def decode(self, tgt_input_ids, enc_out):
        T = tgt_input_ids.shape[1]
        x = self.drop(self.token_emb(tgt_input_ids) + self.dec_pos[:T].unsqueeze(0))
        for block in self.decoder_blocks:
            x = block(x, enc_out)
        return self.lm_head(self.dec_ln_f(x))

    def forward(self, src_ids, tgt_input_ids, tgt_target_ids=None):
        enc_out = self.encode(src_ids)
        logits = self.decode(tgt_input_ids, enc_out)
        loss = None
        if tgt_target_ids is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt_target_ids.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        enc_out = self.encode(src_ids[:, -self.max_src_len:])
        dec_ids = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            logits = self.decode(dec_ids[:, -self.max_tgt_len:], enc_out)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=src_ids.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            dec_ids = torch.cat([dec_ids, next_token], dim=1)
        return dec_ids[:, 1:]

class PrefixAwareAttention(nn.Module):
    """Causal self-attention, except query positions inside the prefix (index < prefix_len)
    also get bidirectional access to the rest of the prefix. Positions >= prefix_len are
    plain causal. A single stack, single attention type per layer - neither a fully causal
    decoder nor a two-stack encoder-decoder."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, prefix_len):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        pos = torch.arange(T, device=x.device)
        in_prefix = pos < prefix_len
        prefix_bonus = in_prefix.unsqueeze(1) & in_prefix.unsqueeze(0)
        allowed = self.causal_mask[:T, :T] | prefix_bonus

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~allowed, float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class PrefixLMBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixAwareAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x, prefix_len):
        x = x + self.attn(self.ln1(x), prefix_len)
        x = x + self.ff(self.ln2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([PrefixLMBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, prefix_len, targets=None, loss_mask=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x, prefix_len)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            per_tok = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), reduction="none").view(B, T)
            m = (torch.arange(T, device=idx.device) + 1 >= prefix_len).to(per_tok.dtype)
            loss = (per_tok * m).sum() / (m.sum() * B).clamp(min=1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, prefix_len, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond, prefix_len=min(prefix_len, idx_cond.shape[1]))
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

class LightBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_heads, n_kv_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x

class LightweightLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            LightBlock(d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class QKNormAttention(nn.Module):
    """Full multi-head causal attention with RoPE, plus QK-Norm: Q and K are each RMSNorm'd
    per head (over the head dimension) before the dot product, bounding attention logit scale
    independent of how large Q/K activations get during training - improves stability."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.d_head)
        self.k_norm = RMSNorm(self.d_head)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

class QKBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = QKNormAttention(d_model, n_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x

class RopeSwiGLUQKNormLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([QKBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

In [4]:
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")

def find_phase1(arch):
    hits = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_model.pt"), recursive=True)
    assert hits, f"{arch}_model.pt not found under {INPUT_ROOT}. Attach the dataset made from your Phase-1 zips."
    d = os.path.dirname(hits[0])
    for n in (f"{arch}_bpe_tokenizer.json", "bpe_tokenizer.json"):
        if os.path.exists(os.path.join(d, n)): return hits[0], os.path.join(d, n)
    t = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_bpe_tokenizer.json"), recursive=True) or glob.glob(os.path.join(INPUT_ROOT, "**", "bpe_tokenizer.json"), recursive=True)
    assert t, "tokenizer json not found next to the model"
    return hits[0], t[0]

def load_phase1_backbone(arch, path):
    ck = torch.load(path, map_location="cpu", weights_only=False); cfg, sd = ck["config"], ck["state_dict"]; g = cfg.get
    if arch == "decoder": m = DecoderLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "encdec": m = EncoderDecoderLM(g("vocab_size"), g("d_model"), g("n_enc_layers"), g("n_dec_layers"), g("n_heads"), g("d_ff"), g("max_src_len"), g("max_tgt_len"), g("dropout", 0.1))
    elif arch == "hybrid": m = PrefixLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "lightweight": m = LightweightLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("n_kv_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "rope_qknorm": m = RopeSwiGLUQKNormLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    else: raise ValueError(arch)
    m.load_state_dict(sd, strict=True)
    return m, cfg, (g("max_seq_len") or g("max_src_len"))

def block_list_name(arch): return "encoder_blocks" if arch == "encdec" else "blocks"

def strip_unused(m, arch):
    m.lm_head = nn.Identity()
    if arch == "encdec": m.decoder_blocks = nn.ModuleList(); m.dec_ln_f = nn.Identity()

def prune_layers(m, arch, keep):
    blocks = getattr(m, block_list_name(arch)); setattr(m, block_list_name(arch), nn.ModuleList([blocks[i] for i in keep]))

In [5]:
class LoRALinear(nn.Module):
    """LoRA: y = Wx + s*B(Ax). rsLoRA = same with s = alpha/sqrt(r) instead of alpha/r (rank-stabilised)."""
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x): return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """LoRMA+ : W' = (I + (alpha/r) B A) W0 (multiplicative); A=0, B=kaiming as in the authors' code, so W'=W0 at step 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.zeros(r, base.out_features)); self.B = nn.Parameter(torch.empty(base.out_features, r))
        nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
    def forward(self, x):
        h = F.linear(x, self.base.weight); y = h + self.s * F.linear(F.linear(h, self.A), self.B)
        return y + self.base.bias if self.base.bias is not None else y

In [6]:
class BPE:
    """Phase-1 byte-level BPE (load + encode). Same merges and special ids as pretraining."""
    SPECIAL = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self):
        self.merges = {}; self.id_to_bytes = {i: bytes([i]) for i in range(256)}; self.special_to_id = {}
        for i, t in enumerate(self.SPECIAL): self.special_to_id[t] = 256 + i; self.id_to_bytes[256 + i] = t.encode()
    @staticmethod
    def _merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair: out.append(new_id); i += 2
            else: out.append(seq[i]); i += 1
        return out
    @classmethod
    def load(cls, path):
        tok = cls()
        for ps, nid in json.load(open(path, encoding="utf-8"))["merges"].items():
            a, b = (int(x) for x in ps.split(",")); tok.merges[(a, b)] = nid; tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok
    def encode_piece(self, piece):
        seq = list(piece.encode("utf-8"))
        while len(seq) > 1:
            cand = [p for p in zip(seq, seq[1:]) if p in self.merges]
            if not cand: break
            pair = min(cand, key=lambda p: self.merges[p]); seq = self._merge(seq, pair, self.merges[pair])
        return seq

class WordTok:
    def __init__(self, bpe):
        sp = bpe.special_to_id
        self.bpe, self.pad, self.eosw, self.sep, self.end, self.cache = bpe, sp["<pad>"], sp["</s>"], sp["<s>"], sp["<unk>"], {}
    def encode_words(self, words):
        ids = []
        for w in words:
            p = self.cache.get(w)
            if p is None: p = self.cache[w] = self.bpe.encode_piece(w) + [self.eosw]
            ids.extend(p)
        return ids
    def decode(self, ids):
        words, cur = [], []
        flush = lambda: words.append(b"".join(self.bpe.id_to_bytes.get(int(x), b"") for x in cur).decode("utf-8", errors="ignore"))
        for t in ids:
            t = int(t)
            if t == self.end: break
            if t == self.eosw:
                if cur: flush(); cur = []
            elif t not in (self.pad, self.sep): cur.append(t)
        if cur: flush()
        return " ".join(words)

def read_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)
def get_path(rel):
    if LOCAL_DATA_DIR: return os.path.join(LOCAL_DATA_DIR, rel)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_REPO, filename=f"{P6_DIR}/{rel}", repo_type="dataset", token=hf_token)

def parse_docs(raw, labelled):
    pairs = list(raw.items()) if isinstance(raw, dict) and "id" not in raw else [(r.get("id", r.get("doc_id")), r) for r in raw]
    out = []
    for did, v in pairs:
        if not isinstance(v, dict): v = {"document": v}
        text = next((v[k] for k in ("document", "text", "judgment", "content") if k in v), None)
        assert text is not None, f"doc {did}: no document field (keys: {list(v)})"
        if isinstance(text, list): text = " ".join(map(str, text))
        summ = None
        if labelled:
            k = next((k for k in ("summary", "summaries", "reference", "abstract") if k in v), None)
            assert k is not None, f"doc {did}: no summary field (keys: {list(v)})"
            summ = " ".join(map(str, v[k])) if isinstance(v[k], list) else str(v[k])
        out.append(dict(id=str(did), document=str(text), summary=summ))
    return sorted(out, key=lambda d: d["id"])

_train_all = parse_docs(read_json(get_path("Train/train.json")), True)
TEST = parse_docs(read_json(get_path("Test/test.json")), False)
_rng = random.Random(SEED); _idx = list(range(len(_train_all))); _rng.shuffle(_idx)
_nd = min(max(DEV_MIN, int(DEV_FRAC * len(_idx))), max(len(_idx) // 5, 1))
DEV = sorted((_train_all[i] for i in _idx[:_nd]), key=lambda d: d["id"]); TRAIN = sorted((_train_all[i] for i in _idx[_nd:]), key=lambda d: d["id"])
for nm, ds in (("train", TRAIN), ("dev", DEV), ("test", TEST)): assert len({d["id"] for d in ds}) == len(ds), f"duplicate ids in {nm}"
if TRAIN_DOC_LIMIT and len(TRAIN) > TRAIN_DOC_LIMIT: TRAIN = random.Random(SEED).sample(TRAIN, TRAIN_DOC_LIMIT)
if SMOKE:
    TRAIN, DEV, TEST = TRAIN[:16], DEV[:6], TEST[:4]; EPOCHS = {k: 1 for k in EPOCHS}; print(">>> SMOKE MODE: tiny subset, 1 epoch <<<")
DOCS = {"train": TRAIN, "dev": DEV, "test": TEST}
for nm, ds in DOCS.items():
    wl = [len(d["document"].split()) for d in ds]; sl = [len(d["summary"].split()) for d in ds] if ds[0]["summary"] is not None else None
    print(f"{nm:5s}: {len(ds):5d} docs | document words median {int(np.median(wl))}, max {max(wl)}" + (f" | summary words median {int(np.median(sl))}, max {max(sl)}" if sl else ""))

SENT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
_df = Counter(); [_df.update(set(d["document"].lower().split())) for d in TRAIN]
IDF = {w: math.log((len(TRAIN) + 1) / (c + 1)) + 1 for w, c in _df.items()}; IDF_DEFAULT = max(IDF.values()) if IDF else 1.0; del _df

def select_input(text, budget, wt, mode):
    words = text.split()
    if mode == "head": return wt.encode_words(words[: budget // 2 + 8])[:budget]
    if mode == "headtail":
        h = budget // 2; a = wt.encode_words(words[: h // 2 + 8])[:h]; b = wt.encode_words(words[-(h // 2 + 8):])[-(budget - len(a)):]
        return a + b
    sents = [s for s in SENT_RE.split(text) if s and s.strip()]
    if len(sents) <= 1: return wt.encode_words(words[: budget // 2 + 8])[:budget]
    tf = Counter(w.lower() for w in words); enc, score = [], []
    for s in sents:
        ws = s.split(); enc.append(wt.encode_words(ws))
        score.append(sum(tf[w.lower()] * IDF.get(w.lower(), IDF_DEFAULT) for w in ws) / (len(ws) + 5))
    chosen, tot = [], 0
    for i in np.argsort(score)[::-1]:
        if tot + len(enc[i]) <= budget: chosen.append(int(i)); tot += len(enc[i])
        if tot >= budget - 8: break
    out = [t for i in sorted(chosen) for t in enc[i]]
    return (out or enc[0])[:budget]

def pack(docs, wt, Ls, Lt, mode):
    N = len(docs); src = np.full((N, Ls), wt.pad, np.int16); tgt = np.full((N, Lt), wt.pad, np.int16)
    for i, d in enumerate(docs):
        ids = select_input(d["document"], Ls, wt, mode); src[i, Ls - len(ids):] = ids
        if d["summary"] is not None:
            t = wt.encode_words(d["summary"].split()[:Lt])[:Lt - 1] + [wt.end]; tgt[i, :len(t)] = t
    return dict(src=torch.from_numpy(src), tgt=torch.from_numpy(tgt))

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

train:  6679 docs | document words median 3131, max 139645 | summary words median 638, max 27768
dev  :   351 docs | document words median 3105, max 68930 | summary words median 638, max 13861
test :   100 docs | document words median 3171, max 39695


In [7]:
class SumModel(nn.Module):
    """Phase-1 language model used as a summariser.
       decoder-style (decoder / hybrid / lightweight / rope_qknorm):  [document tokens][<s>][summary tokens]  -> next-token prediction on the summary part
       encoder-decoder:  encoder reads the document, decoder writes the summary."""
    def __init__(self, arch, m, Ls, sep, pad):
        super().__init__(); self.arch, self.m, self.Ls, self.sep, self.pad = arch, m, Ls, sep, pad
        self.d_model = m.token_emb.weight.shape[1]
    def lm(self, x):
        if self.arch == "hybrid": return self.m(x, self.Ls + 1)[0]
        return self.m(x)[0]
    def forward(self, src, tgt):
        src, tgt = src.long(), tgt.long(); B = tgt.shape[0]; col = torch.full((B, 1), self.sep, dtype=torch.long, device=tgt.device)
        if self.arch == "encdec": return self.m.decode(torch.cat([col, tgt[:, :-1]], 1), self.m.encode(src))
        return self.lm(torch.cat([src, col, tgt[:, :-1]], 1))[:, self.Ls:]

@torch.no_grad()
def generate(sm, src, max_new):
    sm.eval(); src = src.to(DEVICE).long(); B = src.shape[0]; col = torch.full((B, 1), sm.sep, dtype=torch.long, device=DEVICE)
    outs, done, ngr = [[] for _ in range(B)], [False] * B, [{} for _ in range(B)]; n = NO_REPEAT_NGRAM
    if sm.arch == "encdec":
        enc = sm.m.encode(src); x = col
    else: x = torch.cat([src, col], 1)
    for t in range(max_new):
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            lg = (sm.m.decode(x, enc) if sm.arch == "encdec" else sm.lm(x))[:, -1]
        lg = lg.float()
        if t < MIN_NEW: lg[:, WT.end] = -1e9
        for b in range(B):
            if done[b]: continue
            if REP_PENALTY != 1.0 and outs[b]:
                ids = torch.tensor(list(set(outs[b])), device=DEVICE); v = lg[b, ids]; lg[b, ids] = torch.where(v > 0, v / REP_PENALTY, v * REP_PENALTY)
            if len(outs[b]) >= n - 1:
                for tok in ngr[b].get(tuple(outs[b][len(outs[b]) - (n - 1):]), ()): lg[b, tok] = -1e9
        nxt = lg.argmax(-1)
        for b in range(B):
            if done[b]: continue
            tk = int(nxt[b]); outs[b].append(tk)
            if len(outs[b]) >= n: ngr[b].setdefault(tuple(outs[b][-n:-1]), set()).add(tk)
            if tk == WT.end: done[b] = True
        x = torch.cat([x, nxt[:, None]], 1)
        if all(done): break
    return [WT.decode(o) for o in outs]

In [8]:
NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
       -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
       0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]

class QLoRALinear(nn.Module):
    """QLoRA: frozen base weight stored as block-wise 4-bit NF4 (two codes per byte, block=64, absmax scale) + trainable LoRA."""
    def __init__(self, base, r, alpha, block=64):
        super().__init__()
        w = base.weight.detach().float().flatten(); self.n, self.shape, self.block = w.numel(), tuple(base.weight.shape), block
        if (-self.n) % block: w = F.pad(w, (0, (-self.n) % block))
        blocks = w.view(-1, block); absmax = blocks.abs().max(1, keepdim=True).values.clamp(min=1e-8)
        table = torch.tensor(NF4, device=w.device)
        codes = ((blocks / absmax).unsqueeze(-1) - table).abs().argmin(-1).to(torch.uint8)
        self.register_buffer("packed", (codes[:, 0::2] << 4) | codes[:, 1::2]); self.register_buffer("absmax", absmax.squeeze(1))
        self.register_buffer("table", table)
        self.register_buffer("bias", None if base.bias is None else base.bias.detach().clone())
        self.s = alpha / r; self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def dequant(self):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 0xF).long()], -1).view(self.packed.shape[0], -1)
        return (self.table[codes] * self.absmax[:, None]).flatten()[:self.n].view(self.shape)
    def forward(self, x):
        return F.linear(x, self.dequant().to(x.dtype), None if self.bias is None else self.bias.to(x.dtype)) + self.s * F.linear(F.linear(x, self.A), self.B)

class Adapter(nn.Module):
    """Bottleneck adapter; the up-projection starts at zero so the model starts identical to Phase 1."""
    def __init__(self, d, bottleneck):
        super().__init__(); self.down, self.up, self.act = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d), nn.GELU()
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return x + self.up(self.act(self.down(x)))

class Adapted(nn.Module):
    def __init__(self, inner, d, bottleneck): super().__init__(); self.inner, self.adapter = inner, Adapter(d, bottleneck)
    def forward(self, *a, **k): return self.adapter(self.inner(*a, **k))

def block_lists(arch, bb): return [bb.encoder_blocks, bb.decoder_blocks] if arch == "encdec" else [bb.blocks]

def prune_all(bb, arch, keep_every):
    for name in (["encoder_blocks", "decoder_blocks"] if arch == "encdec" else ["blocks"]):
        blocks = getattr(bb, name); setattr(bb, name, nn.ModuleList([blocks[i] for i in range(0, len(blocks), keep_every)]))

def _swap(root, names, make):
    n = 0
    for _, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(mod, cname, make(child)); n += 1
    return n

def apply_method(sm, method):
    bb, P = sm.m, PEFT
    if method in ("full", "kd"):
        for p in sm.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "lora": n = _swap(bb, P["targets"], lambda b: LoRALinear(b, P["r"], P["alpha"]))
        elif method == "lorma": n = _swap(bb, P["lorma_targets"], lambda b: LoRMALinear(b, P["r"], P["alpha"]))
        elif method == "qlora": n = _swap(bb, P["targets"], lambda b: QLoRALinear(b, P["r"], P["alpha"]))
        elif method in ("houlsby", "pfeiffer"):
            names = ("attn", "self_attn", "cross_attn", "ff") if method == "houlsby" else ("ff",); n = 0
            for blocks in block_lists(sm.arch, bb):
                for blk in blocks:
                    for cname, child in list(blk.named_children()):
                        if cname in names: setattr(blk, cname, Adapted(child, sm.d_model, P["bottleneck"])); n += 1
        elif method == "bitfit":
            n = 0
            for nm, p in bb.named_parameters():
                if nm.endswith(".bias"): p.requires_grad = True; n += 1
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                    for p in mod.parameters(): p.requires_grad = True; n += 1
        elif method == "partial":
            n = 0
            for blocks in block_lists(sm.arch, bb):
                for blk in list(blocks)[-P["partial_last"]:]:
                    for p in blk.parameters(): p.requires_grad = True; n += 1
        else: raise ValueError(f"unknown method {method}")
        assert n > 0, f"{method}: nothing to adapt"
    tr = sum(p.numel() for p in sm.parameters() if p.requires_grad); tot = sum(p.numel() for p in sm.parameters())
    print(f"[{method}] trainable {tr/1e6:.3f}M of {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")

def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}

In [9]:
def lcs_length(a, b):
    if not a or not b: return 0
    if len(a) < len(b): a, b = b, a
    masks = {}
    for j, t in enumerate(a): masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0); x = m | state; y = (state << 1) | 1; state = x & ~(x - y)
    return bin(state).count("1")

def rouge(pred, ref):
    """ROUGE-1 / ROUGE-L F1 on lower-cased whitespace tokens (an approximation of the official scorer, used for model selection)."""
    p, r = pred.lower().split(), ref.lower().split()
    if not p or not r: return 0.0, 0.0
    ov = sum((Counter(p) & Counter(r)).values()); l = lcs_length(p, r)
    return 2 * ov / (len(p) + len(r)), 2 * l / (len(p) + len(r))

def tri_rep(text):
    w = text.split(); tg = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    return 0.0 if not tg else 1 - len(set(tg)) / len(tg)

def score_summaries(preds, docs):
    sc = [rouge(preds[d["id"]], d["summary"]) for d in docs]
    out = dict(rouge1=float(np.mean([s[0] for s in sc])), rougeL=float(np.mean([s[1] for s in sc])),
               words_pred=float(np.mean([len(preds[d["id"]].split()) for d in docs])), words_ref=float(np.mean([len(d["summary"].split()) for d in docs])),
               tri_rep=float(np.mean([tri_rep(preds[d["id"]]) for d in docs])), bertscore=None)
    if DEV_BERTSCORE:
        try:
            import bert_score; out["bertscore"] = float(bert_score.score([preds[d["id"]] for d in docs], [d["summary"] for d in docs], lang="en", verbose=False)[2].mean())
        except Exception as e: print("  (BERTScore skipped:", str(e)[:100], ")")
    return out

def build_sum(arch, p1_path, method, keep_every=None):
    bb, cfg, max_len = load_phase1_backbone(arch, p1_path)
    if keep_every: prune_all(bb, arch, keep_every)
    if arch == "encdec": Ls, Lt = SRC_TOK_ED, min(TGT_TOK_ED, cfg["max_tgt_len"] - 1); assert Ls <= cfg["max_src_len"], "source budget exceeds the encoder length"
    else: Ls, Lt = SRC_TOK_LM, TGT_TOK_LM; assert Ls + Lt <= max_len, f"{Ls}+{Lt} exceeds the Phase-1 context {max_len}"
    sm = SumModel(arch, bb, Ls, WT.sep, WT.pad); apply_method(sm, method)
    return sm.to(DEVICE), Ls, Lt

def loss_fn(lg, tgt, t_lg=None):
    m = tgt != WT.pad; s, y = lg[m].float(), tgt[m]
    hard = F.cross_entropy(s, y, label_smoothing=LABEL_SMOOTH)
    if t_lg is None: return hard
    soft = F.kl_div(F.log_softmax(s / KD_T, -1), F.softmax(t_lg[m].float() / KD_T, -1), reduction="batchmean") * KD_T * KD_T
    return KD_ALPHA * hard + (1 - KD_ALPHA) * soft

@torch.no_grad()
def dev_loss(model, pk, bs=8):
    model.eval(); tot = n = 0.0
    for b in range(0, len(pk["src"]), bs):
        s, t = pk["src"][b:b + bs].to(DEVICE), pk["tgt"][b:b + bs].to(DEVICE).long()
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"): lg = model(s, t)
        m = t != WT.pad; tot += F.cross_entropy(lg[m].float(), t[m], reduction="sum").item(); n += m.sum().item()
    return tot / max(n, 1)

def fit(model, pk, ckdir, method, teacher=None, seed=0, dev_pk=None):
    os.makedirs(ckdir, exist_ok=True); last_p, best_p = f"{ckdir}/last.pt", f"{ckdir}/best.pt"
    dm = nn.DataParallel(model) if NUM_GPUS > 1 else model
    tr = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(tr, lr=LR[method], weight_decay=WEIGHT_DECAY); scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    ds = torch.utils.data.TensorDataset(pk["src"], pk["tgt"])
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=len(ds) > BATCH_SIZE, generator=torch.Generator().manual_seed(seed))
    epochs = EPOCHS[method]; spe = max(len(loader) // GRAD_ACCUM, 1); total, warm = epochs * spe, max(int(WARMUP_FRAC * epochs * spe), 1)
    start, best, hist, bad = 0, 1e9, [], 0
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["trainable"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best, hist, bad = ck["epoch"], ck["best"], ck["hist"], ck["bad"]; print(f"resumed at epoch {start}/{epochs}")
    step, t0 = start * spe, time.time()
    for ep in range(start, epochs):
        dm.train(); run = nb = 0; sl = 0.0; te = time.time(); opt.zero_grad(set_to_none=True)
        for i, (s, t) in enumerate(loader):
            s, t = s.to(DEVICE), t.to(DEVICE).long()
            with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
                lg = dm(s, t); tl = None
                if teacher is not None:
                    with torch.no_grad(): tl = teacher(s, t)
                loss = loss_fn(lg, t, tl) / GRAD_ACCUM
            scaler.scale(loss).backward(); lv = loss.item() * GRAD_ACCUM; run += lv; sl += lv; nb += 1
            if (i + 1) % GRAD_ACCUM == 0:
                f = (step + 1) / warm if step < warm else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(total - warm, 1)))
                for g in opt.param_groups: g["lr"] = LR[method] * f
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                avg = sl / GRAD_ACCUM; sl = 0.0
                if LOG_EVERY and step % LOG_EVERY == 0:
                    k = (i + 1) // GRAD_ACCUM; el = time.time() - te
                    print(f"    ep {ep+1}/{epochs} step {k}/{spe} | loss {avg:.4f} | lr {LR[method] * f:.2e} | {el:.0f}s elapsed, epoch ETA {el / k * (spe - k):.0f}s", flush=True)
        dl = dev_loss(model, dev_pk); hist.append(dict(epoch=ep + 1, train_loss=run / max(nb, 1), dev_loss=dl))
        print(f"  epoch {ep+1}/{epochs} | train loss {run/max(nb,1):.4f} | dev loss {dl:.4f} (ppl {math.exp(min(dl, 20)):.1f}) | {(time.time()-t0)/60:.1f} min", flush=True)
        if dl < best: best, bad = dl, 0; torch.save({"trainable": trainable_state(model)}, best_p + ".tmp"); os.replace(best_p + ".tmp", best_p)
        else: bad += 1
        torch.save({"trainable": trainable_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "epoch": ep + 1, "best": best, "hist": hist, "bad": bad}, last_p + ".tmp"); os.replace(last_p + ".tmp", last_p)
        if bad >= PATIENCE: print("  early stop (patience)"); break
        if RUN_MAX_MIN and (time.time() - t0) / 60 > RUN_MAX_MIN: print("  stopping: time limit for this run"); break
    model.load_state_dict(torch.load(best_p, map_location="cpu", weights_only=False)["trainable"], strict=False)
    return hist, best

In [10]:
def setup_arch(arch):
    global ARCH, WORK, P1_MODEL, WT, TOK_SHA, PACKS
    ARCH = arch; WORK = os.path.join(OUT_ROOT, f"{EXP_TAG}_{arch}"); os.makedirs(WORK, exist_ok=True); PACKS = {}
    P1_MODEL, p1_tok = find_phase1(arch); WT = WordTok(BPE.load(p1_tok))
    TOK_SHA = hashlib.sha256(open(p1_tok, "rb").read()).hexdigest(); print(f"\n##### {arch}: model {P1_MODEL} | tokenizer {TOK_SHA[:10]}")
def get_packs(Ls, Lt, mode):
    if (Ls, Lt, mode) not in PACKS: PACKS[(Ls, Lt, mode)] = {sp: pack(DOCS[sp], WT, Ls, Lt, mode) for sp in DOCS}
    return PACKS[(Ls, Lt, mode)]

def gen_all(sm, pk, docs, Lt):
    out = {}
    for b in range(0, len(docs), GEN_BATCH):
        res = generate(sm, pk["src"][b:b + GEN_BATCH], Lt - 1)
        for d, s in zip(docs[b:b + GEN_BATCH], res): out[d["id"]] = s
        print(f"    generated {min(b + GEN_BATCH, len(docs))}/{len(docs)}", flush=True)
    return out

def write_submission(summaries, name, outdir):
    sub = [{"id": d["id"], "summary": summaries[d["id"]]} for d in TEST]; assert len(sub) == len(TEST)
    pj = f"{outdir}/prediction.json"; json.dump(sub, open(pj, "w"), ensure_ascii=False); zp = f"{OUT_ROOT}/{name}_submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
    return zp

def run_one(spec):
    method = spec["method"]; name = spec.get("name", method); mode = spec.get("mode", INPUT_MODE); outdir = f"{WORK}/{name}"; os.makedirs(outdir, exist_ok=True)
    if os.path.exists(f"{outdir}/done.json"):
        print(f"[{ARCH}/{name}] already done -> skipped"); zp = f"{OUT_ROOT}/{EXP_TAG}_{ARCH}_{name}_submission.zip"
        if not os.path.exists(zp) and os.path.exists(f"{outdir}/prediction.json"):
            with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(f"{outdir}/prediction.json", arcname="prediction.json")
        return json.load(open(f"{outdir}/done.json"))
    print(f"\n{'='*20} {ARCH} / {name}  (input={mode}) {'='*20}"); t_start = time.time()
    model, Ls, Lt = build_sum(ARCH, P1_MODEL, method, STUDENT_KEEP_EVERY if method == "kd" else None); pk = get_packs(Ls, Lt, mode)
    teacher = None
    if method == "kd":
        cands = [m for m in (json.load(open(p)) for p in glob.glob(f"{WORK}/*/done.json")) if m["method"] != "kd" and m["mode"] == mode]
        assert cands, "kd needs a finished non-kd run of this model (same input mode) earlier in its list"
        tm = max(cands, key=lambda m: m["dev_rougeL"]); print(f"  teacher = {tm['name']} (dev ROUGE-L {tm['dev_rougeL']:.4f}); student keeps every {STUDENT_KEEP_EVERY}nd block")
        teacher, _, _ = build_sum(ARCH, P1_MODEL, tm["method"])
        teacher.load_state_dict(torch.load(f"{WORK}/{tm['name']}/ckpt/best.pt", map_location="cpu", weights_only=False)["trainable"], strict=False)
        teacher.eval(); [p.requires_grad_(False) for p in teacher.parameters()]
    hist, best = fit(model, pk["train"], f"{outdir}/ckpt", method, teacher, dev_pk=pk["dev"])
    print("  generating dev summaries ..."); dev_s = gen_all(model, pk["dev"], DEV, Lt); sc = score_summaries(dev_s, DEV)
    print("  generating test summaries ..."); test_s = gen_all(model, pk["test"], TEST, Lt)
    json.dump(dev_s, open(f"{outdir}/dev_summaries.json", "w"), ensure_ascii=False); json.dump(test_s, open(f"{outdir}/test_summaries.json", "w"), ensure_ascii=False)
    meta = dict(task="phase6_summ", name=name, arch=ARCH, method=method, mode=mode, tag=EXP_TAG, tokenizer_sha=TOK_SHA, src_tok=Ls, tgt_tok=Lt, dev_loss=best,
                dev_rouge1=sc["rouge1"], dev_rougeL=sc["rougeL"], dev_words_pred=sc["words_pred"], dev_words_ref=sc["words_ref"], dev_tri_rep=sc["tri_rep"], dev_bertscore=sc["bertscore"],
                trainable_M=round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3), total_M=round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
                lr=LR[method], epochs_run=len(hist), minutes=round((time.time() - t_start) / 60, 1), smoke=SMOKE)
    json.dump(meta, open(f"{outdir}/meta.json", "w"), indent=1); zp = write_submission(test_s, f"{EXP_TAG}_{ARCH}_{name}", outdir)
    print(f"  dev: ROUGE-1 {sc['rouge1']:.4f} | ROUGE-L {sc['rougeL']:.4f} | words {sc['words_pred']:.0f} (ref {sc['words_ref']:.0f}) | repeated-3gram {sc['tri_rep']:.3f}" + (f" | BERTScore {sc['bertscore']:.4f}" if sc["bertscore"] else "") + f"\n  submission: {zp}")
    json.dump(meta, open(f"{outdir}/done.json", "w")); del model, teacher; gc.collect(); torch.cuda.empty_cache()
    return meta

def zip_dir(zpath, files):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
        for f, arc in files:
            if os.path.exists(f): z.write(f, arc)
    print(f"{os.path.getsize(zpath)/1e6:.1f} MB  {zpath}")

RESULTS = []
for arch, runs in PLAN:
    setup_arch(arch)
    for s in runs: RESULTS.append(run_one(s))
    done = [os.path.dirname(p) for p in glob.glob(f"{WORK}/*/done.json")]
    zip_dir(f"{OUT_ROOT}/{EXP_TAG}_{arch}_bundle.zip", [(f"{d}/{n}", f"{os.path.basename(WORK)}/{os.path.basename(d)}/{n}") for d in done for n in ("dev_summaries.json", "test_summaries.json", "meta.json")])
print("\nSESSION SUMMARY"); [print(f"  {m['arch']:11s} {m['name']:9s} ROUGE-1 {m['dev_rouge1']:.4f} | ROUGE-L {m['dev_rougeL']:.4f} | {m['minutes']} min") for m in RESULTS]
dirs = [os.path.join(OUT_ROOT, d) for d in os.listdir(OUT_ROOT) if d.startswith(f"{EXP_TAG}_") and os.path.isdir(os.path.join(OUT_ROOT, d))]
zip_dir(f"{OUT_ROOT}/{EXP_TAG}_ALL_outputs.zip", [(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), OUT_ROOT)) for d in dirs for dp, _, fs in os.walk(d) for f in fs if f != "last.pt"])
print("Download the *_submission.zip files and the ALL_outputs zip from the Output panel BEFORE closing. Do NOT use Save Version / Save & Run All.")


##### lightweight: model /kaggle/input/datasets/radynx/lightweight/lightweight_model.pt | tokenizer 8623378cb3

==================== lightweight / pfeiffer  (input=select) ====================
[pfeiffer] trainable 1.586M of 111.49M (1.42%)
    ep 1/6 step 25/417 | loss 2.4998 | lr 8.33e-05 | 17s elapsed, epoch ETA 269s
    ep 1/6 step 50/417 | loss 2.3665 | lr 1.67e-04 | 32s elapsed, epoch ETA 238s
    ep 1/6 step 75/417 | loss 2.2454 | lr 2.50e-04 | 48s elapsed, epoch ETA 217s
    ep 1/6 step 100/417 | loss 2.1649 | lr 3.33e-04 | 63s elapsed, epoch ETA 200s
    ep 1/6 step 125/417 | loss 2.2461 | lr 4.17e-04 | 79s elapsed, epoch ETA 184s
    ep 1/6 step 150/417 | loss 2.2156 | lr 5.00e-04 | 94s elapsed, epoch ETA 168s
    ep 1/6 step 175/417 | loss 2.1837 | lr 5.00e-04 | 110s elapsed, epoch ETA 152s
    ep 1/6 step 200/417 | loss 2.1881 | lr 5.00e-04 | 126s elapsed, epoch ETA 137s
    ep 1/6 step 225/417 | loss 2.2260 | lr 4.99e-04 | 142s elapsed, epoch ETA 121s
    ep 1/6 step 250/4

In [11]:
ENSEMBLE = True
if ENSEMBLE:
    metas = []
    for root in (OUT_ROOT, INPUT_ROOT):
        for mp in glob.glob(os.path.join(root, "**", "meta.json"), recursive=True):
            try: m = json.load(open(mp))
            except Exception: continue
            if m.get("task") == "phase6_summ" and not m.get("smoke") and os.path.exists(os.path.join(os.path.dirname(mp), "test_summaries.json")):
                m["_dir"] = os.path.dirname(mp); metas.append(m)
    seen = {}
    for m in metas: seen.setdefault((m["arch"], m["name"]), m)
    metas = sorted(seen.values(), key=lambda m: -m["dev_rougeL"]); assert metas, "no finished runs found"
    DS = [json.load(open(m["_dir"] + "/dev_summaries.json")) for m in metas]; TS = [json.load(open(m["_dir"] + "/test_summaries.json")) for m in metas]
    print(f"{len(metas)} runs found:"); [print(f"  {m['arch']:11s} {m['name']:9s} dev ROUGE-1 {m['dev_rouge1']:.4f} | ROUGE-L {m['dev_rougeL']:.4f}") for m in metas]
    def mbr(cands):
        if len(cands) < 3: return cands[0]
        sc = [np.mean([rouge(c, o)[1] for j, o in enumerate(cands) if j != i]) for i, c in enumerate(cands)]
        return cands[int(np.argmax(sc))]
    def finish(idx, name):
        dev_s = {d["id"]: mbr([DS[i][d["id"]] for i in idx]) for d in DEV}; sc = score_summaries(dev_s, DEV)
        test_s = {d["id"]: mbr([TS[i][d["id"]] for i in idx]) for d in TEST}
        od = f"{OUT_ROOT}/{EXP_TAG}_ensemble_{name}"; os.makedirs(od, exist_ok=True); zp = write_submission(test_s, f"{EXP_TAG}_ensemble_{name}", od)
        print(f"  {name:6s} members={len(idx)} dev ROUGE-1 {sc['rouge1']:.4f} | ROUGE-L {sc['rougeL']:.4f} | words {sc['words_pred']:.0f} -> {zp}")
    finish(list(range(len(metas))), "mbr_all"); finish(list(range(min(3, len(metas)))), "mbr_top3")

2 runs found:
  lightweight kd        dev ROUGE-1 0.0433 | ROUGE-L 0.0367
  lightweight pfeiffer  dev ROUGE-1 0.0403 | ROUGE-L 0.0352
  mbr_all members=2 dev ROUGE-1 0.0433 | ROUGE-L 0.0367 | words 20 -> /kaggle/working/p6_ensemble_mbr_all_submission.zip
  mbr_top3 members=2 dev ROUGE-1 0.0433 | ROUGE-L 0.0367 | words 20 -> /kaggle/working/p6_ensemble_mbr_top3_submission.zip
